In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
!pip install datasets duckdb pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 37.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.21.3 requires pyarrow<20,>=4.0.0, but you have pyarrow 23.0.1 which is incompatible.


In [3]:
!pip install sentence-transformers torch

## Load FinMTEB dataset (FinSTS)

In [5]:
from datasets import load_dataset

dataset_name = "FinanceMTEB/FinSTS"

ds = load_dataset(dataset_name)

print(ds)
print("Splits:", ds.keys())

DatasetDict({
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 370
    })
})
Splits: dict_keys(['test'])


## Inspect the dataset

In [6]:
# pick first split (usually 'test')
split_name = list(ds.keys())[0]

print("Using split:", split_name)

# show one record
print(ds[split_name][0])

Using split: test
{'sentence1': 'We have undertaken, or will undertake, a variety of initiatives to integrate, standardize, centralize, and streamline our operations with technology, including, but not limited to, an enterprise resource planning system and a customer information and billing system.', 'sentence2': 'We have undertaken, and will continue to undertake, a variety of initiatives to integrate, standardize, centralize, and streamline our operations with technology, including, but not limited to, the implementation of several different ERP systems.', 'score': 1}


In [7]:
# Checking Columns

print("Columns:", ds[split_name].column_names)
print("Number of rows:", len(ds[split_name]))

Columns: ['sentence1', 'sentence2', 'score']
Number of rows: 370


In [8]:
# Converting to pandas

df = ds[split_name].to_pandas()
df.head()

,sentence1,sentence2,score
0,"We have undertaken, or will undertake, a varie...","We have undertaken, and will continue to under...",1
1,"Any failures, negative publicity or lawsuits r...","Any failures, negative publicity or lawsuits r...",1
2,The Company depends on operations in internati...,The Company depends on operations in internati...,1
3,"As of June 30, 2018, we owned or exclusively l...","As of June 30, 2019, we owned or exclusively l...",0
4,A federal court has recently determined that t...,Several federal courts have recently issued de...,1


# Save raw dataset locally

### This gives you a frozen copy in case Hugging Face changes something or internet is flaky on JupyterHub.

In [9]:
import os
os.makedirs("data/raw", exist_ok=True)

file_path = f"data/raw/finsts_{split_name}.parquet"

df.to_parquet(file_path, index=False)

print("Saved to:", file_path)

Saved to: data/raw/finsts_test.parquet


### Create DuckDB database (inside notebook)

In [10]:
import duckdb

con = duckdb.connect("finmteb.duckdb")
print("Database created!")

Database created!


## Create table

In [12]:
# Metadata table

con.execute("""
CREATE TABLE IF NOT EXISTS datasets_metadata (
    dataset_name TEXT,
    split_name TEXT,
    num_rows INTEGER,
    columns TEXT
)
""")

In [14]:
# Insert metadat

con.execute("""
INSERT INTO datasets_metadata VALUES (?, ?, ?, ?)
""", [
    dataset_name,
    split_name,
    len(df),
    ", ".join(df.columns.tolist())
])


In [15]:
# Raw data table

con.execute("""
CREATE TABLE IF NOT EXISTS records_raw (
    dataset_name TEXT,
    split_name TEXT,
    record_id TEXT,
    text_a TEXT,
    text_b TEXT,
    label TEXT
)
""")

## Map dataset columns

In [16]:
print(df.columns.tolist())

['sentence1', 'sentence2', 'score']


In [17]:
text_a_col = "sentence1"
text_b_col = "sentence2"
label_col = "score"

## Insert first records into database

In [19]:
sample_df = df.head(20).copy()

In [20]:
# create sample first
sample_df = df.head(20).copy()

rows = []

for i, row in sample_df.iterrows():
    rows.append((
        dataset_name,
        split_name,
        str(i),
        str(row[text_a_col]),
        str(row[text_b_col]),
        str(row[label_col])
    ))

con.executemany("""
INSERT INTO records_raw VALUES (?, ?, ?, ?, ?, ?)
""", rows)

print("Inserted 20 rows!")

Inserted 20 rows!


## Check Dataset

In [21]:
con.execute("SELECT * FROM records_raw LIMIT 5").fetchdf()

,dataset_name,split_name,record_id,text_a,text_b,label
0,FinanceMTEB/FinSTS,test,0,"We have undertaken, or will undertake, a varie...","We have undertaken, and will continue to under...",1
1,FinanceMTEB/FinSTS,test,1,"Any failures, negative publicity or lawsuits r...","Any failures, negative publicity or lawsuits r...",1
2,FinanceMTEB/FinSTS,test,2,The Company depends on operations in internati...,The Company depends on operations in internati...,1
3,FinanceMTEB/FinSTS,test,3,"As of June 30, 2018, we owned or exclusively l...","As of June 30, 2019, we owned or exclusively l...",0
4,FinanceMTEB/FinSTS,test,4,A federal court has recently determined that t...,Several federal courts have recently issued de...,1


In [22]:
con.execute("SELECT COUNT(*) FROM records_raw").fetchall()

[(20,)]

In [23]:
con.execute("SELECT * FROM datasets_metadata").fetchdf()

,dataset_name,split_name,num_rows,columns
0,FinanceMTEB/FinSTS,test,370,"sentence1, sentence2, score"


### Insert full Dataset

In [24]:
rows = []

for i, row in df.iterrows():
    rows.append((
        dataset_name,
        split_name,
        str(i),
        str(row[text_a_col]),
        str(row[text_b_col]),
        str(row[label_col])
    ))

con.executemany("""
INSERT INTO records_raw VALUES (?, ?, ?, ?, ?, ?)
""", rows)

print("Inserted full dataset!")

Inserted full dataset!


### Now we have 

1. Loaded a real EMNLP benchmark dataset
2. Explored schema
3. Saved raw data
4.  a database
5. Inserted structured records

### now Let's do the simplest possible baseline run:

In [25]:
# installing model libraries in notebook

!pip install sentence-transformers scikit-learn

In [26]:
# reopen your DuckDB database

import duckdb
import pandas as pd

con = duckdb.connect("finmteb.duckdb")

In [27]:
# Read stored records

df_db = con.execute("""
SELECT record_id, text_a, text_b, label
FROM records_raw
WHERE text_a IS NOT NULL AND text_b IS NOT NULL
LIMIT 100
""").fetchdf()

df_db.head()

,record_id,text_a,text_b,label
0,0,"We have undertaken, or will undertake, a varie...","We have undertaken, and will continue to under...",1
1,1,"Any failures, negative publicity or lawsuits r...","Any failures, negative publicity or lawsuits r...",1
2,2,The Company depends on operations in internati...,The Company depends on operations in internati...,1
3,3,"As of June 30, 2018, we owned or exclusively l...","As of June 30, 2019, we owned or exclusively l...",0
4,4,A federal court has recently determined that t...,Several federal courts have recently issued de...,1


#### load an embedding model - Use a small strong baseline first:

In [28]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
import torch
print(torch.cuda.is_available())

True


In [30]:
# Generate embeddings

texts_a = df_db["text_a"].tolist()
texts_b = df_db["text_b"].tolist()

emb_a = model.encode(texts_a, convert_to_tensor=True, show_progress_bar=True)
emb_b = model.encode(texts_b, convert_to_tensor=True, show_progress_bar=True)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [31]:
# compute cosine similarity

from sentence_transformers import util

cos_scores = util.cos_sim(emb_a, emb_b).diagonal().cpu().numpy()
df_db["pred_score"] = cos_scores
df_db.head()

,record_id,text_a,text_b,label,pred_score
0,0,"We have undertaken, or will undertake, a varie...","We have undertaken, and will continue to under...",1,0.870419
1,1,"Any failures, negative publicity or lawsuits r...","Any failures, negative publicity or lawsuits r...",1,0.940167
2,2,The Company depends on operations in internati...,The Company depends on operations in internati...,1,0.902322
3,3,"As of June 30, 2018, we owned or exclusively l...","As of June 30, 2019, we owned or exclusively l...",0,0.860113
4,4,A federal court has recently determined that t...,Several federal courts have recently issued de...,1,0.894108


In [32]:
# compare predictions with labels - For STS-style tasks, labels are usually similarity scores, so use correlation.

df_db["label"] = pd.to_numeric(df_db["label"], errors="coerce")
eval_df = df_db.dropna(subset=["label", "pred_score"]).copy()

print("Rows used for evaluation:", len(eval_df))

Rows used for evaluation: 100


In [34]:
# Now compute Pearson and Spearman:

from scipy.stats import pearsonr, spearmanr

pearson_corr, _ = pearsonr(eval_df["label"], eval_df["pred_score"])
spearman_corr, _ = spearmanr(eval_df["label"], eval_df["pred_score"])

print("Pearson:", pearson_corr)
print("Spearman:", spearman_corr)

Pearson: 0.1743184267936665
Spearman: 0.19886071194358504


In [35]:
# create an evaluation table in DuckDB

con.execute("""
CREATE TABLE IF NOT EXISTS evaluation_runs (
    run_id BIGINT,
    model_name TEXT,
    dataset_name TEXT,
    split_name TEXT,
    metric_name TEXT,
    metric_value DOUBLE,
    num_rows INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

In [36]:
# insert your metrics

dataset_used = con.execute("""
SELECT dataset_name, split_name
FROM records_raw
LIMIT 1
""").fetchone()

dataset_name_db, split_name_db = dataset_used

run_id = con.execute("""
SELECT COALESCE(MAX(run_id), 0) + 1 FROM evaluation_runs
""").fetchone()[0]

rows_to_insert = [
    (run_id, model_name, dataset_name_db, split_name_db, "pearson", float(pearson_corr), int(len(eval_df))),
    (run_id, model_name, dataset_name_db, split_name_db, "spearman", float(spearman_corr), int(len(eval_df)))
]

con.executemany("""
INSERT INTO evaluation_runs
(run_id, model_name, dataset_name, split_name, metric_name, metric_value, num_rows)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", rows_to_insert)

print("Saved evaluation run:", run_id)

Saved evaluation run: 1


In [37]:
# Verify saved result

con.execute("""
SELECT *
FROM evaluation_runs
ORDER BY run_id DESC
""").fetchdf()

,run_id,model_name,dataset_name,split_name,metric_name,metric_value,num_rows,created_at
0,1,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,pearson,0.174318,100,2026-03-22 22:12:45.549053
1,1,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,spearman,0.198861,100,2026-03-22 22:12:45.822703


### now we have a complete working NLP pipeline:

1. dataset loaded

2. stored in DB

3. model run

4. metrics computed

5. results saved

### Let’s interpret result

we got:

Pearson: 0.174

Spearman: 0.198

A general-purpose embedding model (all-MiniLM-L6-v2) achieved low correlation scores (Spearman ≈ 0.20) on the FinSTS dataset, indicating limited domain transfer to financial text. This highlights the need for domain-specific models such as Fin-E5 proposed in the paper.

## Compare 3 models cleanly

In [40]:
!pip install -q sentence-transformers scipy scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [41]:
import duckdb
import pandas as pd
import torch
import gc
from sentence_transformers import SentenceTransformer, util
from scipy.stats import pearsonr, spearmanr

con = duckdb.connect("finmteb.duckdb")

df_db = con.execute("""
SELECT record_id, text_a, text_b, label, dataset_name, split_name
FROM records_raw
WHERE text_a IS NOT NULL AND text_b IS NOT NULL
""").fetchdf()

df_db["label"] = pd.to_numeric(df_db["label"], errors="coerce")
df_db = df_db.dropna(subset=["label"]).copy()

print("Rows:", len(df_db))
print("Label range:", df_db["label"].min(), df_db["label"].max())
print("CUDA available:", torch.cuda.is_available())

Rows: 390
Label range: 0 1
CUDA available: True


In [42]:
if df_db["label"].max() > 1:
    df_db["label"] = df_db["label"] / df_db["label"].max()

In [43]:
# Use these models

models = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-base-v2",
    "BAAI/bge-base-en-v1.5",
]

In [44]:
def prepare_texts(model_name, texts, side):
    if "e5" in model_name.lower():
        prefix = "query: " if side == "a" else "passage: "
        return [prefix + str(t) for t in texts]
    return [str(t) for t in texts]

#### Run each model

In [51]:
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_model(model_name, df, batch_size=128):
    print(f"\nRunning: {model_name}")
    model = SentenceTransformer(model_name, device="cuda")
    print("Device:", model.device)

    texts_a = prepare_texts(model_name, df["text_a"].tolist(), side="a")
    texts_b = prepare_texts(model_name, df["text_b"].tolist(), side="b")

    emb_a = model.encode(
        texts_a,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    emb_b = model.encode(
        texts_b,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    pred = util.cos_sim(emb_a, emb_b).diagonal().cpu().numpy()

    out = df.copy()
    out["pred_score"] = pred

    # 🔥 FIX: convert label to numeric
    out["label"] = pd.to_numeric(out["label"], errors="coerce")

    # drop invalid rows
    out = out.dropna(subset=["label", "pred_score"])

    pearson_corr, _ = pearsonr(out["label"], out["pred_score"])
    spearman_corr, _ = spearmanr(out["label"], out["pred_score"])

    cleanup()

    return {
        "model_name": model_name,
        "pearson": float(pearson_corr),
        "spearman": float(spearman_corr),
        "num_rows": int(len(out))
    }

In [54]:
results = []

for model_name in models:
    metrics = run_model(model_name, df_db, batch_size=128)
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("spearman", ascending=False)
results_df


Running: sentence-transformers/all-MiniLM-L6-v2
Device: cuda:0


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Running: intfloat/e5-base-v2
Device: cuda:0


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Running: BAAI/bge-base-en-v1.5
Device: cuda:0


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

,model_name,pearson,spearman,num_rows
2,BAAI/bge-base-en-v1.5,0.217502,0.219596,390
0,sentence-transformers/all-MiniLM-L6-v2,0.186436,0.210987,390
1,intfloat/e5-base-v2,0.122383,0.104154,390


In [55]:
# Save the results:

con.execute("""
CREATE TABLE IF NOT EXISTS evaluation_runs (
    run_id BIGINT,
    model_name TEXT,
    dataset_name TEXT,
    split_name TEXT,
    metric_name TEXT,
    metric_value DOUBLE,
    num_rows INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

run_base = con.execute("""
SELECT COALESCE(MAX(run_id), 0) + 1 FROM evaluation_runs
""").fetchone()[0]

dataset_name_db = df_db["dataset_name"].iloc[0]
split_name_db = df_db["split_name"].iloc[0]

rows_to_insert = []
for i, row in results_df.reset_index(drop=True).iterrows():
    run_id = run_base + i
    rows_to_insert.append((
        run_id, row["model_name"], dataset_name_db, split_name_db,
        "pearson", row["pearson"], row["num_rows"]
    ))
    rows_to_insert.append((
        run_id, row["model_name"], dataset_name_db, split_name_db,
        "spearman", row["spearman"], row["num_rows"]
    ))

con.executemany("""
INSERT INTO evaluation_runs
(run_id, model_name, dataset_name, split_name, metric_name, metric_value, num_rows)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", rows_to_insert)

con.execute("""
SELECT *
FROM evaluation_runs
ORDER BY run_id DESC, metric_name
""").fetchdf()

,run_id,model_name,dataset_name,split_name,metric_name,metric_value,num_rows,created_at
0,7,intfloat/e5-base-v2,FinanceMTEB/FinSTS,test,pearson,0.122383,390,2026-03-22 22:49:26.169212
1,7,intfloat/e5-base-v2,FinanceMTEB/FinSTS,test,spearman,0.104154,390,2026-03-22 22:49:26.206477
2,6,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,pearson,0.186436,390,2026-03-22 22:49:26.086303
3,6,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,spearman,0.210987,390,2026-03-22 22:49:26.133152
4,5,BAAI/bge-base-en-v1.5,FinanceMTEB/FinSTS,test,pearson,0.217502,390,2026-03-22 22:49:25.950315
5,5,BAAI/bge-base-en-v1.5,FinanceMTEB/FinSTS,test,spearman,0.219596,390,2026-03-22 22:49:26.034702
6,4,intfloat/e5-base-v2,FinanceMTEB/FinSTS,test,pearson,0.122383,390,2026-03-22 22:39:33.027215
7,4,intfloat/e5-base-v2,FinanceMTEB/FinSTS,test,spearman,0.104154,390,2026-03-22 22:39:33.080509
8,3,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,pearson,0.186436,390,2026-03-22 22:39:32.868097
9,3,sentence-transformers/all-MiniLM-L6-v2,FinanceMTEB/FinSTS,test,spearman,0.210987,390,2026-03-22 22:39:32.963582


## Run full dataset on GPU

In [56]:
# Use the full saved split:

df_db = con.execute("""
SELECT record_id, text_a, text_b, label, dataset_name, split_name
FROM records_raw
WHERE text_a IS NOT NULL AND text_b IS NOT NULL
""").fetchdf()

In [57]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
print(model.device)

cuda:0


In [58]:
metrics = run_model(model_name, df_db, batch_size=64)


Running: BAAI/bge-base-en-v1.5
Device: cuda:0


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

### Use the official benchmark repo

In [60]:
!git clone https://github.com/yixuantt/FinMTEB.git
!pip install -q -r FinMTEB/requirements.txt

fatal: destination path 'FinMTEB' already exists and is not an empty directory.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [62]:
# Add a sparse baseline - Use TF-IDF as your sparse baseline:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from scipy.stats import pearsonr, spearmanr

# convert labels to numeric first
df_db["label"] = pd.to_numeric(df_db["label"], errors="coerce")
tfidf_eval_df = df_db.dropna(subset=["label"]).copy()

texts_a = df_db["text_a"].astype(str).tolist()
texts_b = df_db["text_b"].astype(str).tolist()

all_texts = texts_a + texts_b
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
X = vectorizer.fit_transform(all_texts)

Xa = X[:len(texts_a)]
Xb = X[len(texts_a):]

tfidf_scores = np.array([
    cosine_similarity(Xa[i], Xb[i])[0, 0]
    for i in range(len(texts_a))
])

pearson_tfidf, _ = pearsonr(df_db["label"], tfidf_scores)
spearman_tfidf, _ = spearmanr(df_db["label"], tfidf_scores)

print("TF-IDF Pearson:", pearson_tfidf)
print("TF-IDF Spearman:", spearman_tfidf)



TF-IDF Pearson: 0.2424767602736803
TF-IDF Spearman: 0.2690051142611229


In [64]:
print(tfidf_eval_df["label"].min(), tfidf_eval_df["label"].max())

0 1


### Your Final Results (Clean Comparison)
A.  Dense Models
Model:                                     Pearson	   &       Spearman
1. BAAI/bge-base-en-v1.5 :                 Pearson 0.2175	   & Spearman      0.2196
3. sentence-transformers/all-MiniLM-L6-v2:	Pearson 0.1864	   &  Spearman     0.2110
4. intfloat/e5-base-v2:	                    Pearson 0.1224	   & Spearman       0.1042

B. Sparse Baseline
Model :	       -Pearson	 &  -Spearman
TF-IDF (BoW): Pearson	0.2443	 & Spearman  0.2690

## Model Comparison on FinSTS

##### Models Evaluated:

1. MiniLM

2. E5-base

3. BGE-base

4. TF-IDF (baseline)

##### Results:

Best dense model: BGE (Spearman ≈ 0.22)
TF-IDF outperformed all models (≈ 0.27)

##### Key Insight:
Financial similarity tasks are lexical-heavy
Simple methods can outperform deep models

##### Conclusion:
General embeddings ≠ domain performance
Domain-specific models (FinE5) are needed

## Key Points

1. TF-IDF beats all neural models

Our result:
TF-IDF Spearman = 0.269

Best dense model (BGE) = 0.219

This is a VERY IMPORTANT finding - This matches the paper’s claim:

Financial STS tasks are lexical-heavy → simple Bag-of-Words can outperform dense embeddings

We just experimentally confirmed a core result of the paper


2. BGE performs best among dense models

BGE > MiniLM > E5

Interpretation:

BGE is stronger at semantic similarity

MiniLM is lightweight but competitive 

E5 underperformed here → likely due to: needing better prompt formatting or not being domain-adapted

3. Domain mismatch problem

All our models are general-purpose

The paper shows:

Finance-specific model (FinE5) performs better

So our result supports:

“General embedding models do not transfer well to financial domain tasks”

#### Our improvement from earlier run

Earlier:

~100 rows

Now:

390 rows (full dataset)

This makes our results: more reliable

closer to benchmark evaluation

## REPORT PARAGRAPH (you can copy this)

Here’s a polished version for your submission:

In this experiment, we evaluated three general-purpose embedding models—sentence-transformers/all-MiniLM-L6-v2, intfloat/e5-base-v2, and BAAI/bge-base-en-v1.5—on the FinanceMTEB FinSTS dataset using cosine similarity and correlation metrics. Among the dense models, BGE achieved the highest performance (Spearman ≈ 0.22), followed by MiniLM (≈ 0.21), while E5 performed significantly worse (≈ 0.10).

Interestingly, a sparse baseline using TF-IDF outperformed all dense models, achieving a Spearman correlation of approximately 0.27. This result aligns with findings reported in the FinMTEB paper, where lexical-based methods can outperform neural embeddings on financial semantic similarity tasks. These results suggest that financial text similarity relies heavily on surface-level lexical overlap rather than deep semantic abstraction, highlighting the limitations of general-purpose embedding models in domain-specific settings.

This experiment demonstrates that benchmark performance on general datasets does not necessarily translate to domain-specific tasks, reinforcing the importance of domain-adapted evaluation frameworks such as FinMTEB.

# We now have:

1. reproducible pipeline

2. database-backed evaluation

3. multi-model comparison

4. baseline + advanced models

5. results aligned with paper